# Web Sockets

Normally, in HTTP (the usual request–response model), the client sends a request, the server replies, and the connection ends.

👉 But sometimes we need real-time communication (like chat apps, live notifications, dashboards).
For that, WebSockets allow a persistent two-way connection between client and server.

So:
- HTTP → One-time request & response
- WebSocket → Continuous open channel (both sides can send messages anytime)

## How WebSockets Work
1. Client asks the server: “Can we upgrade this HTTP connection to WebSocket?”
2. If server agrees, they create a persistent connection.
3. Now both client and server can send messages to each other anytime without reopening connections.

Think of it like a phone call (WebSocket) vs sending letters (HTTP).

In [ ]:
from fastapi import FastAPI, WebSocket
from fastapi.responses import HTMLResponse

app = FastAPI()

html = """
<!DOCTYPE html>
<html>
    <head>
        <title>WebSocket Example</title>
    </head>
    <body>
        <h1>WebSocket Test</h1>
        <input id="messageText" type="text" placeholder="Type a message"/>
        <button onclick="sendMessage()">Send</button>
        <ul id="messages"></ul>

        <script>
            var ws = new WebSocket("ws://localhost:8000/ws");
            
            ws.onmessage = function(event) {
                var messages = document.getElementById('messages')
                var message = document.createElement('li')
                message.textContent = event.data
                messages.appendChild(message)
            };

            function sendMessage() {
                var input = document.getElementById("messageText")
                ws.send(input.value)
                input.value = ''
            }
        </script>
    </body>
</html>
"""

@app.get("/")
async def get():
    return HTMLResponse(html)

@app.websocket("/ws")
async def websocket_endpoint(websocket: WebSocket):
    await websocket.accept()  # Accept connection
    while True:
        data = await websocket.receive_text()  # Receive message
        await websocket.send_text(f"You wrote: {data}")  # Send response


**What happens here?**

1. Open browser → `http://localhost:8000/`
2. It shows a text box + button.
3. When you type and click Send:
    - The message goes from client → server via WebSocket.
    - Server replies with `You wrote: <message>`.
    - Browser displays it in the list.

✅ That’s a real-time 2-way chat between your browser and FastAPI server.

In [ ]:
from fastapi import FastAPI, WebSocket, WebSocketDisconnect

app = FastAPI()

clients = []  # list of connected clients

@app.websocket("/ws")
async def websocket_endpoint(websocket: WebSocket):
    await websocket.accept()
    clients.append(websocket)
    try:
        while True:
            data = await websocket.receive_text()
            # Broadcast message to all clients
            for client in clients:
                await client.send_text(f"Client says: {data}")
    except WebSocketDisconnect:
        clients.remove(websocket)


Open: 👉 `http://127.0.0.1:8000/`

Open it in two browser tabs → type messages → you’ll see the broadcast in both tabs.

**What happens here?**
- Multiple users connect.
- When one sends a message, everyone receives it.
- This is like a group chat app.

## When to Use WebSockets?

- Chat applications
- Real-time notifications (e.g., new order in e-commerce dashboard)
- Live dashboards (stock prices, IoT data, game score updates)
- Collaborative tools (Google Docs style live editing)

## WebSocket tasks in the background

In [ ]:
from sqlalchemy import create_engine, Column, Integer, String, Text
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker

DATABASE_URL = "sqlite:///./chat.db"

engine = create_engine(DATABASE_URL, connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(bind=engine, autoflush=False, autocommit=False)

Base = declarative_base()

class Message(Base):
    __tablename__ = "messages"
    id = Column(Integer, primary_key=True, index=True)
    content = Column(Text, nullable=False)

Base.metadata.create_all(bind=engine)


In [ ]:
from fastapi import FastAPI, WebSocket, WebSocketDisconnect, Depends
from sqlalchemy.orm import Session
import asyncio

app = FastAPI()

# Reuse DB session
def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

clients = []  # connected clients list

@app.websocket("/ws")
async def websocket_endpoint(websocket: WebSocket, db: Session = Depends(get_db)):
    await websocket.accept()
    clients.append(websocket)
    try:
        while True:
            data = await websocket.receive_text()

            # 1️⃣ Save message in background
            asyncio.create_task(save_message(db, data))

            # 2️⃣ Broadcast message to all clients
            for client in clients:
                await client.send_text(f"User says: {data}")

    except WebSocketDisconnect:
        clients.remove(websocket)


async def save_message(db: Session, message: str):
    """Save message asynchronously into DB"""
    msg = Message(content=message)
    db.add(msg)
    db.commit()


**What happens here?**

1. User sends a message via WebSocket.
2. Server does two things:
    - Immediately broadcasts it to all users → ensures real-time response.
    - In background, saves it to the DB → ensures persistence.
3. Even if saving to DB is slow, the chat isn’t blocked.

## Retrieve old chat history

**Step 1: Update WebSocket Endpoint to Send History**

We’ll query the database for stored messages and send them to the client right after they connect.

In [ ]:
from fastapi import FastAPI, WebSocket, WebSocketDisconnect, Depends
from sqlalchemy.orm import Session
import asyncio

app = FastAPI()

clients = []  # connected clients list

# DB session dependency
def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()


@app.websocket("/ws")
async def websocket_endpoint(websocket: WebSocket, db: Session = Depends(get_db)):
    await websocket.accept()
    clients.append(websocket)

    # 1️⃣ Send chat history on connection
    messages = db.query(Message).all()
    for msg in messages:
        await websocket.send_text(f"[History] {msg.content}")

    try:
        while True:
            data = await websocket.receive_text()

            # 2️⃣ Save message in background
            asyncio.create_task(save_message(data))

            # 3️⃣ Broadcast new message
            for client in clients:
                await client.send_text(f"User says: {data}")

    except WebSocketDisconnect:
        clients.remove(websocket)


async def save_message(message: str):
    """Save message asynchronously into DB"""
    db = SessionLocal()
    try:
        msg = Message(content=message)
        db.add(msg)
        db.commit()
    finally:
        db.close()


**Step 2: What Changed?**

1. When a new client connects:
    - Server loads all stored messages from DB.
- Sends them to that client with a [History] tag.
    - Now the user sees past chat before starting new messages.

2. New messages are still:
    - Saved to DB in background.
    - Broadcasted to all connected clients in real time.

**Step 3: Example User Flow**
1. User A joins first → empty screen (no old messages yet).
2. User A types: “Hello!” → sent, saved, broadcasted.
3. User B joins later → instantly receives: `[History] Hello!`

## Separate users with usernames

**Step 1: Modify Frontend (index.html)**

We’ll ask the user for a username before connecting to the WebSocket:

In [ ]:
# index.html:

<!DOCTYPE html>
<html>
    <head>
        <title>WebSocket Chat</title>
    </head>
    <body>
        <h1>WebSocket Chat</h1>
        
        <input id="username" type="text" placeholder="Enter your username"/>
        <button onclick="connect()">Join Chat</button>
        
        <div id="chat" style="display:none;">
            <input id="messageText" type="text" placeholder="Type a message"/>
            <button onclick="sendMessage()">Send</button>
            <ul id="messages"></ul>
        </div>

        <script>
            let ws;
            let username;

            function connect() {
                username = document.getElementById("username").value;
                if (!username) {
                    alert("Enter a username first!");
                    return;
                }

                ws = new WebSocket("ws://localhost:8000/ws/" + username);

                ws.onmessage = function(event) {
                    var messages = document.getElementById('messages')
                    var message = document.createElement('li')
                    message.textContent = event.data
                    messages.appendChild(message)
                };

                document.getElementById("chat").style.display = "block";
            }

            function sendMessage() {
                var input = document.getElementById("messageText")
                ws.send(input.value)
                input.value = ''
            }
        </script>
    </body>
</html>


**Step 2: Update Backend (FastAPI)**

We’ll capture the username from the WebSocket path and include it in the messages.

In [ ]:
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from sqlalchemy.orm import Session
import asyncio

app = FastAPI()
clients = []  # list of (websocket, username)


@app.websocket("/ws/{username}")
async def websocket_endpoint(websocket: WebSocket, username: str):
    await websocket.accept()
    clients.append((websocket, username))

    # 1️⃣ Send chat history on connection
    db = SessionLocal()
    messages = db.query(Message).all()
    for msg in messages:
        await websocket.send_text(f"[History] {msg.content}")
    db.close()

    try:
        while True:
            data = await websocket.receive_text()

            # 2️⃣ Save message with username
            message_text = f"{username}: {data}"
            asyncio.create_task(save_message(message_text))

            # 3️⃣ Broadcast to all
            for client, user in clients:
                await client.send_text(message_text)

    except WebSocketDisconnect:
        clients.remove((websocket, username))


async def save_message(message: str):
    """Save message asynchronously into DB"""
    db = SessionLocal()
    try:
        msg = Message(content=message)
        db.add(msg)
        db.commit()
    finally:
        db.close()


## Add private messaging

**Step 1: Update Backend (FastAPI)**

We’ll keep track of clients in a dictionary `{username: websocket}` so we can send directly.

In [ ]:
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from sqlalchemy.orm import Session
import asyncio

app = FastAPI()

# Dictionary of connected users
clients = {}  # {username: websocket}


@app.websocket("/ws/{username}")
async def websocket_endpoint(websocket: WebSocket, username: str):
    await websocket.accept()
    clients[username] = websocket

    # 1️⃣ Send chat history on connection
    db = SessionLocal()
    messages = db.query(Message).all()
    for msg in messages:
        await websocket.send_text(f"[History] {msg.content}")
    db.close()

    try:
        while True:
            data = await websocket.receive_text()

            # If message starts with @ -> private DM
            if data.startswith("@"):
                target, msg = data.split(" ", 1)
                target_user = target[1:]  # remove '@'

                if target_user in clients:
                    private_msg = f"(Private) {username} -> {target_user}: {msg}"
                    await clients[target_user].send_text(private_msg)
                    await websocket.send_text(private_msg)  # sender sees it too
                    asyncio.create_task(save_message(private_msg))
                else:
                    await websocket.send_text(f"User '{target_user}' not online")

            else:
                # Public message
                message_text = f"{username}: {data}"
                asyncio.create_task(save_message(message_text))

                for user, client_ws in clients.items():
                    await client_ws.send_text(message_text)

    except WebSocketDisconnect:
        del clients[username]


async def save_message(message: str):
    """Save message asynchronously into DB"""
    db = SessionLocal()
    try:
        msg = Message(content=message)
        db.add(msg)
        db.commit()
    finally:
        db.close()


**Step 2: Frontend (index.html)**

No changes needed 🚀 — users just type @username message to send a private message.
For example: `@Bob Hello, this is private!`

# Modular Application Structure

It’s a way to organize your FastAPI project into smaller, reusable, and maintainable parts (modules) instead of one big file.

Each module (like `users`, `products`, `auth`, `orders`) has:
- its own routes (APIs)
- its own schemas/models
- sometimes database logic
- sometimes services

This structure keeps code clean, scalable, and easier to debug.

In [ ]:
app/
│── main.py
│── core/                # core config (settings, db connection, security)
│── users/
│   │── router.py        # all user routes
│   │── schemas.py       # pydantic models for users
│   │── models.py        # database models (if using SQLAlchemy)
│── products/
│   │── router.py
│   │── schemas.py
│   │── models.py


**Here’s a common structure you’ll find in large projects:**

In [ ]:
app/
│── main.py                # Entry point of application
│── core/                  # Core settings & config
│   │── config.py          # Environment variables, app settings
│   │── security.py        # Auth/security utils (JWT, hashing)
│   │── db.py              # Database connection setup
│   │── middleware.py      # Custom middlewares
│
│── api/                   # All API endpoints grouped by modules
│   │── v1/                # Versioning your API
│   │   │── router.py      # Root router for v1
│   │   │── users.py       # Users endpoints
│   │   │── products.py    # Products endpoints
│
│── models/                # SQLAlchemy models
│   │── user.py
│   │── product.py
│
│── schemas/               # Pydantic models
│   │── user.py
│   │── product.py
│
│── services/              # Business logic (separated from routes)
│   │── user_service.py
│   │── product_service.py
│
│── dependencies/          # ⭐ Dependencies go here
│   │── db.py              # DB session dependency
│   │── auth.py            # Auth/JWT dependencies
│   │── common.py          # Other shared deps
│    
│── utils/                 # Helper functions (e.g. email, logging)
│   │── email.py
│   │── logger.py
│
│── tests/                 # Unit and integration tests
│   │── test_users.py
│   │── test_products.py


## Services

In a well-structured FastAPI app, the services/ layer is where you put your business logic — things that:
- Combine multiple CRUD actions
- Handle workflows (e.g., creating an order and adjusting inventory)
- Validate complex conditions
- Send emails, trigger events, etc.- 

### Example

In [ ]:
# services/user_service.py:

# (Service layer = business logic)

from sqlalchemy.orm import Session
from models.user import User
from schemas.user import UserCreate
from core.security import hash_password

def create_user(db: Session, user_data: UserCreate):
    hashed_pw = hash_password(user_data.password)
    new_user = User(username=user_data.username, email=user_data.email, hashed_password=hashed_pw)
    db.add(new_user)
    db.commit()
    db.refresh(new_user)
    return new_user

def get_user_by_username(db: Session, username: str):
    return db.query(User).filter(User.username == username).first()

def list_users(db: Session):
    return db.query(User).all()


We have one more similar strcuture:

In [ ]:
ecommerce/
├── app/
│   ├── main.py                # Application entrypoint
│   ├── core/                  # Global configs and setup
│   │   ├── config.py          # Settings via Pydantic
│   │   └── database.py        # DB engine, session setup
│   ├── models/                # ORM models per domain
│   │   └── user.py
│   │   └── product.py
│   │   └── order.py
│   ├── schemas/               # Pydantic models for validation
│   │   └── user.py
│   │   └── product.py
│   │   └── order.py
│   ├── crud/                  # Database CRUD operations
│   │   └── user.py
│   │   └── product.py
│   │   └── order.py
│   ├── api/                   # Routers and API versions
│   │   ├── deps.py            # Shared dependencies (e.g., DB session)
│   │   └── v1/                # Versioned API (v1)
│   │       └── routes/
│   │           ├── users.py
│   │           ├── products.py
│   │           ├── orders.py
│   └── services/              # Business logic layer
│       └── user_service.py
│       └── product_service.py
│       └── order_service.py
├── tests/                     # Test modules
│   └── test_users.py
│   └── test_products.py
│   └── test_orders.py
├── alembic/                   # Database migrations (Alembic)
├── requirements.txt
├── .env
└── README.md


# Admin Panel

## Tortoise ORM + Aerich + FastAPI-Admin

- FastAPI-Admin is a package inspired by Django Admin.
- Works with `Tortoise ORM`.
- Provides UI to manage database models.


`pip install fastapi fastapi-admin tortoise-orm jinja2 uvicorn`

In [ ]:
# config.py:

# config.py
DATABASE_URL = "sqlite://db.sqlite3"

TORTOISE_ORM = {
    "connections": {"default": DATABASE_URL},
    "apps": {
        "models": {
            "models": ["models", "aerich.models"],  # Include our models
            "default_connection": "default",
        },
    },
}


In [ ]:
# models.py:

from tortoise import fields
from tortoise.models import Model

class User(Model):
    id = fields.IntField(pk=True)
    username = fields.CharField(max_length=50, unique=True)
    is_active = fields.BooleanField(default=True)
    created_at = fields.DatetimeField(auto_now_add=True)

    def __str__(self):
        return self.username


class Blog(Model):
    id = fields.IntField(pk=True)
    title = fields.CharField(max_length=200)
    content = fields.TextField()
    author = fields.ForeignKeyField("models.User", related_name="blogs")
    created_at = fields.DatetimeField(auto_now_add=True)

    def __str__(self):
        return self.title


In [ ]:
# main.py:

from fastapi import FastAPI
from fastapi_admin.app import app as admin_app
from fastapi_admin.providers.login import UsernamePasswordProvider
from fastapi_admin.resources import Model
from fastapi_admin.file_upload import FileUpload
from starlette.requests import Request

from tortoise import Tortoise
from tortoise.contrib.fastapi import register_tortoise

from models import User, Blog
import config


app = FastAPI()


@app.on_event("startup")
async def startup():
    # Init Tortoise ORM
    await Tortoise.init(config=config.TORTOISE_ORM)

    # Configure Admin Panel
    await admin_app.configure(
        logo_url="https://fastapi.tiangolo.com/img/logo-margin/logo-teal.png",
        providers=[
            UsernamePasswordProvider(
                admin_model=User,  # Our User model
                login_logo_url="https://fastapi-admin.github.io/img/logo.png",
            )
        ],
        resources=[
            Model(User),
            Model(Blog),
        ],
    )


# Mount admin app
app.mount("/admin", admin_app)

# Register ORM
register_tortoise(app, config=config.TORTOISE_ORM, generate_schemas=True, add_exception_handlers=True)


Now run the app by `uvicorn main:app --reload` and Go to 👉 `http://127.0.0.1:8000/admin`

### add authentication for superusers only

In [ ]:
# models.py:

from tortoise import fields
from tortoise.models import Model

class User(Model):
    id = fields.IntField(pk=True)
    username = fields.CharField(max_length=50, unique=True)
    password = fields.CharField(max_length=128)  # hashed password
    is_active = fields.BooleanField(default=True)
    is_superuser = fields.BooleanField(default=False)  # ✅ only superusers can access admin
    created_at = fields.DatetimeField(auto_now_add=True)

    def __str__(self):
        return self.username


We override `UsernamePasswordProvider` to check for `is_superuser`.

In [ ]:
# auth_provider.py
from fastapi_admin.providers.login import UsernamePasswordProvider
from starlette.requests import Request
from passlib.context import CryptContext
from models import User

pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")


class SuperuserLoginProvider(UsernamePasswordProvider):
    async def authenticate(self, request: Request, data: dict):
        username = data.get("username")
        password = data.get("password")

        user = await User.get_or_none(username=username)
        if not user:
            return None

        # Verify password
        if not pwd_context.verify(password, user.password):
            return None

        # ✅ Allow only superusers
        if not user.is_superuser:
            return None

        return user


In [ ]:
# main.py:

from fastapi import FastAPI
from fastapi_admin.app import app as admin_app
from fastapi_admin.resources import Model
from tortoise import Tortoise
from tortoise.contrib.fastapi import register_tortoise

from models import User, Blog
import config
from auth_provider import SuperuserLoginProvider

app = FastAPI()


@app.on_event("startup")
async def startup():
    await Tortoise.init(config=config.TORTOISE_ORM)

    # ✅ Only superusers can log in now
    await admin_app.configure(
        logo_url="https://fastapi.tiangolo.com/img/logo-margin/logo-teal.png",
        providers=[SuperuserLoginProvider(admin_model=User)],
        resources=[
            Model(User),
            Model(Blog),
        ],
    )

app.mount("/admin", admin_app)

register_tortoise(app, config=config.TORTOISE_ORM, generate_schemas=True, add_exception_handlers=True)


**Create a superuser**

We’ll hash the password when creating the user. Example script:

In [ ]:
# create_superuser.py
import asyncio
from models import User
from config import TORTOISE_ORM
from tortoise import Tortoise
from passlib.context import CryptContext

pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")


async def create_superuser():
    await Tortoise.init(config=TORTOISE_ORM)
    await Tortoise.generate_schemas()

    password_hash = pwd_context.hash("admin123")
    await User.create(username="admin", password=password_hash, is_superuser=True)
    print("Superuser created: username=admin, password=admin123")

    await Tortoise.close_connections()


if __name__ == "__main__":
    asyncio.run(create_superuser())


Run it: `python create_superuser.py`

Go to `http://127.0.0.1:8000/admin`

Login with:
- Username: admin
- Password: admin123

✅ Only superusers can log in.
Normal users (even if active) won’t be allowed.


### add role-based permissions in Piccolo Admin

In [ ]:
# models.py:

from tortoise import fields
from tortoise.models import Model

class User(Model):
    id = fields.IntField(pk=True)
    username = fields.CharField(max_length=50, unique=True)
    password = fields.CharField(max_length=128)  # hashed password
    is_active = fields.BooleanField(default=True)
    is_superuser = fields.BooleanField(default=False)  
    role = fields.CharField(max_length=20, default="editor")  # ✅ new field: editor / superuser
    created_at = fields.DatetimeField(auto_now_add=True)

    def __str__(self):
        return self.username


class Blog(Model):
    id = fields.IntField(pk=True)
    title = fields.CharField(max_length=200)
    content = fields.TextField()
    author = fields.ForeignKeyField("models.User", related_name="blogs")
    created_at = fields.DatetimeField(auto_now_add=True)

    def __str__(self):
        return self.title


Currently, your `SuperuserLoginProvider` only allows superusers.
Let’s create a new provider that allows both superusers and editors to log in:

In [ ]:
# auth_provider.py:

from fastapi_admin.providers.login import UsernamePasswordProvider
from starlette.requests import Request
from passlib.context import CryptContext
from models import User

pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")


class RoleBasedLoginProvider(UsernamePasswordProvider):
    async def authenticate(self, request: Request, data: dict):
        username = data.get("username")
        password = data.get("password")

        user = await User.get_or_none(username=username)
        if not user:
            return None

        if not pwd_context.verify(password, user.password):
            return None

        # ✅ Only active users with role
        if not user.is_active:
            return None

        return user  # let admin_app handle resource filtering


In [ ]:
# main.py:

from fastapi import FastAPI
from fastapi_admin.app import app as admin_app
from fastapi_admin.resources import Model
from fastapi_admin.depends import get_current_admin
from tortoise import Tortoise
from tortoise.contrib.fastapi import register_tortoise

from models import User, Blog
import config
from auth_provider import RoleBasedLoginProvider

app = FastAPI()


@app.on_event("startup")
async def startup():
    await Tortoise.init(config=config.TORTOISE_ORM)

    async def get_resources(admin):
        """Return resources based on user role"""
        if admin.is_superuser or admin.role == "superuser":
            return [Model(User), Model(Blog)]
        elif admin.role == "editor":
            return [Model(Blog)]
        else:
            return []

    await admin_app.configure(
        logo_url="https://fastapi.tiangolo.com/img/logo-margin/logo-teal.png",
        providers=[RoleBasedLoginProvider(admin_model=User)],
        resources=get_resources,  # ✅ dynamic resources
    )

app.mount("/admin", admin_app)

register_tortoise(app, config=config.TORTOISE_ORM, generate_schemas=True, add_exception_handlers=True)


**Create an editor user**

Add a script like `create_editor.py`:

In [ ]:
# create_editor.py:

import asyncio
from models import User
from config import TORTOISE_ORM
from tortoise import Tortoise
from passlib.context import CryptContext

pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")


async def create_editor():
    await Tortoise.init(config=TORTOISE_ORM)
    await Tortoise.generate_schemas()

    password_hash = pwd_context.hash("editor123")
    await User.create(username="editor", password=password_hash, role="editor", is_superuser=False)
    print("Editor created: username=editor, password=editor123")

    await Tortoise.close_connections()


if __name__ == "__main__":
    asyncio.run(create_editor())


Run it: `python create_editor.py`

- Login with admin / admin123 → you see Users + Blogs.
- Login with editor / editor123 → you see only Blogs.


## SQLModel / SQLAlchemy + Piccolo Admin

- Piccolo Admin works with `SQLAlchemy` & `SQLModel`.
- Has a React-based frontend.
- Provides CRUD functionality.

`pip install fastapi uvicorn piccolo[all]`

In [ ]:
fastapi_piccolo_admin/
│── main.py
│── piccolo_app.py
│── piccolo_conf.py
│── tables.py


In [ ]:
# tables.py:

from piccolo.table import Table
from piccolo.columns import Varchar, Boolean, Timestamptz, Text, ForeignKey

class User(Table):
    username = Varchar(length=50, unique=True)
    password = Varchar(length=200)  # hashed
    is_active = Boolean(default=True)
    is_superuser = Boolean(default=False)
    created_at = Timestamptz()

class Blog(Table):
    title = Varchar(length=200)
    content = Text()
    author = ForeignKey(references=User)
    created_at = Timestamptz()


# This is like defining models in Django ORM.

In [ ]:
# piccolo_conf.py:

from piccolo.conf.apps import AppRegistry
from piccolo.engine.sqlite import SQLiteEngine

DB = SQLiteEngine(path="db.sqlite3")

APP_REGISTRY = AppRegistry(apps=[
    "tables"  # our app
])


# This tells Piccolo which DB and tables to use.

In [ ]:
# piccolo_app.py:

from piccolo_admin.endpoints import create_admin
from tables import User, Blog

admin = create_admin(
    tables=[User, Blog],  # ✅ Show in admin panel
    site_name="FastAPI + Piccolo Admin",
)


# This is where we tell Piccolo Admin which tables should appear.

In [ ]:
# main.py:

from fastapi import FastAPI
from piccolo.engine import engine_finder
from piccolo.apps.user.tables import BaseUser

from piccolo_app import admin
from tables import User, Blog

app = FastAPI()

# Mount Piccolo Admin at /admin
app.mount("/admin", admin)


# Mount Piccolo Admin as a FastAPI route.

**Run migrations**

Piccolo uses its own migration system.
- command: `piccolo migrations new tables --auto`
- command: `piccolo migrations forwards tables`

This creates `db.sqlite3` with `User` and `Blog`.

<br>

**Create a superuser**

Piccolo has built-in user management.
- command: `piccolo user create --username admin --password admin123 --is_superuser`

<br>

**Run the app**
- command `uvicorn main:app --reload`

Go to 👉 `http://127.0.0.1:8000/admin`

### add role-based permissions in Piccolo Admin

In [ ]:
# tables.py:

from piccolo.apps.user.tables import BaseUser
from piccolo.table import Table
from piccolo.columns import Varchar, Text, ForeignKey, Timestamptz


class Blog(Table):
    title = Varchar(length=200)
    content = Text()
    author = ForeignKey(references=BaseUser)  # ✅ link to Piccolo’s user system
    created_at = Timestamptz()


In [ ]:
# piccolo_app.py:

from piccolo_admin.endpoints import create_admin
from piccolo.apps.user.tables import BaseUser
from tables import Blog

admin = create_admin(
    tables=[BaseUser, Blog],  # Show both tables in admin
    site_name="FastAPI + Piccolo Admin",
    allowed_hosts="*",  # allow from anywhere (change in prod)
)


**Migrations**

Run:
- `piccolo migrations new all --auto`
- `piccolo migrations forwards all`

This creates `BaseUser` and `Blog` tables in the database.

<br>

**Create roles & users**

Piccolo allows roles (Admin, Editor, etc.).

- Create a superuser (full access):
    - `piccolo user create --username admin --password admin123 --is_superuser`

- Create an editor (limited access):
    - `piccolo user create --username editor --password editor123`
 
<br>

**Assign permissions**

Piccolo supports per-table permissions (`read`, `create`, `update`, `delete`).

For example, let’s allow the `editor` user to manage only `Blog`: `piccolo user grant editor Blog read create update delete`


Now revoke `User` management permissions for `editor`: `piccolo user revoke editor BaseUser read create update delete`


<br>


**Run the app**
- `uvicorn main:app --reload`


Visit 👉 `http://127.0.0.1:8000/admin`
- Login with `admin/admin123` → Can manage Users + Blogs.
- Login with `editor/editor123` → Can only manage Blogs (Users tab won’t be available).

# Resturant API

## database.py

In [ ]:
# database.py:

from motor.motor_asyncio import AsyncIOMotorClient
from pymongo import ASCENDING
import os
from dotenv import load_dotenv

load_dotenv()                            # Load .env file

MONGO_URL = os.getenv("MONGO_URI")       # fetch connection url from environment

client = AsyncIOMotorClient(MONGO_URL)   # create connection to mongoDB server to perform opeartion

db = client["ecommerce_db"]              # database 

product_collection = db.products         # collection for storing product details
order_collection = db.orders             # collection for storing order details 


**What is `motor`?**
- motor is the asynchronous Python driver for MongoDB.
- It's built on top of `PyMongo` but supports `async` and `await` syntax, making it ideal for async frameworks like FastAPI, Starlette, and Tornado.

**AsyncIOMotorClient**
- This is the async equivalent of `pymongo.MongoClient`.
- It creates a client connection to your MongoDB server, allowing async operations like `.find()`, `.insert_one()`, etc.
- Non-blocking I/O: other FastAPI routes aren't blocked while DB operations are happening.

## schemas.py

In [ ]:
# schemas.py:

from typing import List, Optional
from pydantic import BaseModel, Field, field_validator, model_validator
from bson import ObjectId


class SizeItem(BaseModel): # used by ProductCreate model for sizes
    
    size: str            # pydantic model fields
    quantity: int        # pydantic model fields
    
    @field_validator("quantity") # validating quantity for negative value
    @classmethod
    def quantity_non_negative(cls, value):
        if value < 0:
            raise ValueError("Quantity must be >= 0")
        return value

- Attributes `size`, `quantity` are not traditional `class variables` or `instance variables` in the normal Python sense. Instead, in Pydantic (and dataclasses too), these are `Pydantic model fields`.

In [ ]:
# POST products/
class ProductCreate(BaseModel): # used in POST products/ for validaing data from user for creating new product
    name: str
    price: float
    sizes: List[SizeItem]
    
    @field_validator("price")
    @classmethod
    def price_must_be_positive(cls, v):
        if v <= 0:
            raise ValueError("Price must be greater than 0")
        return v
    
    @model_validator(mode="after") # validating model for duplicate size
    # model-level validator that runs after all field validations
    def validate_unique_sizes(self): # Here 'self' refers to the ProductCreate instance.   
        seen_sizes = set()
        for item in self.sizes:    # self.sizes is already validated as List[SizeItem]
            if item.size in seen_sizes:
                raise ValueError(f"Duplicate size found: {item.size}")
            seen_sizes.add(item.size)
        
        return self # ✅ always return self in mode="after"

        # Rule of thumb:
        #     mode="after" → Always return self (or a modified instance).
        #     mode="before" → Return a dict of data instead, because at that stage the model isn’t built yet.



class ProductResponse(BaseModel): # used in POST products/ as response model for validating returned response
    id: str

- `ProductCreate` is a `Pydantic model` (inherits from `BaseModel`) used for data validation when creating a new product.

**Why use `@model_validator(mode="after")` here?**

Because we're validating the relationship between items in a list, not a single field. Field-level validators (`@field_validator`) work on one field only. This validator checks the entire list for duplicates after field types have already been validated.

In [ ]:
# GET products/
class ProductSummary(BaseModel): # used in GET products/ requesto validate product details to be shown in reponse
    id: str
    name: str
    price: float

class ProductListResponse(BaseModel): # used in GET products/ to validate response data
    data: List[ProductSummary]
    page: dict


In [ ]:
# POST orders/
class OrderItemInput(BaseModel):
    productId: str
    qty: int

    @field_validator("qty")
    @classmethod
    def quantity_must_be_positive(cls, v):
        if v <= 0:
            raise ValueError("Quantity must be greater than 0")
        return v


class OrderCreate(BaseModel): # used in POST orders/ request for validating data coming from user 
    userId: str
    items: List[OrderItemInput]


class OrderResponse(BaseModel): # used in POST orders/ for validating response
    id: str


In [ ]:
# GET orders/<user_id>
class ProductDetails(BaseModel):
    name: str
    id: str

class OrderItemOut(BaseModel):
    productDetails: ProductDetails
    qty: int

class OrderOut(BaseModel):
    id: str
    items: List[OrderItemOut]
    total: float

class PaginationInfo(BaseModel):
    next: int
    limit: int
    previous: int

class OrderListResponse(BaseModel): # used in GET orders/<user_id> for validating response
    data: List[OrderOut]
    page: PaginationInfo 

## main.py

In [ ]:
# main.py:

from fastapi import FastAPI, Query, HTTPException, Path
from typing import List, Optional
from bson import ObjectId

from database import product_collection, order_collection
from schemas import ProductCreate, ProductResponse, ProductListResponse, ProductSummary, OrderCreate, OrderResponse, OrderListResponse

app = FastAPI()

# Create Product API
@app.post("/products", response_model=ProductResponse, status_code=201)
async def create_product(product: ProductCreate):

    print(product)        # name='Cap' price=199.0 sizes=[SizeItem(size='regular', quantity=200)]
    print(type(product))  # <class 'schemas.ProductCreate'>
    
    product_dict = product.model_dump() # convert Pydantic object product into dictionary.
    result = await product_collection.insert_one(product_dict) 

    print(result)          #output:  InsertOneResult(ObjectId('688fa09185c7b3813efc1b43'), acknowledged=True)
    print(type(result))    #output:  <class 'pymongo.results.InsertOneResult'>
    
    return {"id": str(result.inserted_id)} # # should only includes what's defined in ProductResponse


1. When someone sends a `POST` request with a product, this function `create_product()` will be called. FastAPI will automatically parse and validate the request body as a `ProductCreate` model (defined in `schemas.py`).
2. `create_product()` expects a request body matching the schema `ProductCreate` (which includes `name`, `price`, and `sizes`).
    - So `product` is validated by `ProductCreate` schema and return a Pydantic object.
3. `model_dump()` converts the validated Pydantic object product into a plain Python dictionary. Because `motor` (your async MongoDB driver) expects regular dictionaries to insert into MongoDB. This format is needed for inserting into MongoDB.
4. `insert_one()` is an async MongoDB method that inserts a document and returns an object type `InsertOneResult` that includes `inserted_id`.
5. `inserted_id` is then converted into `str` and then `response_model=ProductResponse` → The output/response will follow the `ProductResponse` schema.

**Why are we doing `str(result.inserted_id)`?**

MongoDB stores data in BSON(Binary JSON) format. When documents are stored in MongoDB, their IDs (`_id`) are usually of type `ObjectId` and is unique identifier.

Example: `ObjectId("64c77f44fe45d8a1e7f7a9b1")`

It's not directly serializable to JSON, which means you can't return it in an API response as-is — it must be converted to a string. However, FastAPI (or any JSON-based API) cannot serialize `ObjectId` directly, so you need to transform it into a plain string.

**JSON only allows the following types:**

✅ Allowed: `string`, `number`, `boolean`, `null`, `array`, `object (dictionary)`

❌ Not allowed: `Python datetime objects`, `MongoDB ObjectId`, `Python custom classes`

| Term              | Meaning                                                       |
| ----------------- | ------------------------------------------------------------- |
| Serializable      | Can be converted to a format like JSON                        |
| JSON-serializable | Can be encoded into JSON (string, number, etc.)               |
| ObjectId          | Special MongoDB type, **not** serializable into JSON directly |
| 

In [ ]:
# List Products API
@app.get("/products", response_model=ProductListResponse)
async def list_products(
    name: Optional[str] = None, # filter product by name
    size: Optional[str] = None, # filter product by size
    limit: int = 10,
    offset: int = 0
):
    query = {}
    if name:
        query["name"] = {"$regex": name, "$options": "i"} # If a name is provided, filter products whose name contains the string (case-insensitive)
    if size:
        query["sizes.size"] = size  # search inside embedded sizes
    
    # /products?name=sneakers
    print(query) # {'name': {'$regex': 'sneakers', '$options': 'i'}}
    
    cursor = product_collection.find(query).sort("_id").skip(offset).limit(limit)
    print(cursor) # AsyncIOMotorCursor(<pymongo.synchronous.cursor.Cursor object at 0x000001C05C9BB8E0>)
    print(type(cursor)) # <class 'motor.motor_asyncio.AsyncIOMotorCursor'>
    
    products = [] # store list of Pydantic model instances (ProductSummary)
    
    async for doc in cursor:
        products.append(ProductSummary( 
            id=str(doc["_id"]),
            name=doc["name"],
            price=doc["price"]
        ))

    print(products) # [ProductSummary(id='687b71363e94c1afb4710e3f', name='Sneakers', price=2299.0)]
    print(type(products[0]))  # <class 'schemas.ProductSummary'>
    
    # Pagination logic
    page = {
        "next": offset + limit, # This tells the offset value for the next page
        "limit": len(products), # products returned on this page
        "previous": offset - limit if offset - limit >= 0 else -10 # This calculates the offset to go one page back
    }

    return {"data": products, "page": page} # should only includes what's defined in ProductListResponse

# FastAPI (via Pydantic) automatically converts (serializes) Python objects like ProductSummary into JSON before sending the response to the client.
# You don’t need to manually convert Pydantic models to JSON — FastAPI does that for you under the hood.

- The parameters `name`, `size`, `limit`, and `offset` in the `list_products()` function are query parameters. When you send a `GET` request to the `/products` endpoint, these values can be passed through the URL. `/products?name=sneakers`, FastAPI automatically extracts those query parameters and passes them to the `list_products()` function:
    - `name = "sneakers"`

- MongoDB's `.find()` uses logical AND for multiple keys. So both `name` and `sizes.size` must match in the same document.

- `ProductSummary` converts each MongoDB document into a `ProductSummary` object (only `id`, `name`, and `price`) and appended in `products` list.

- `return {"data": products, "page": page}`: It sees `response_model=ProductListResponse`.

**Why are we returning `products` directly but the API response is supposed to be in JSON?**


- Before sending the response:
    - FastAPI checks if the return value matches the schema.
    - It recursively converts all Pydantic model instances into dictionaries using `.dict()` or `.model_dump()` under the hood.
- Then it serializes everything into JSON to return it as an HTTP response.

FastAPI uses Pydantic's `.model_dump()` or equivalent under the hood to serialize the data to JSON format.

**We can do like this also:**

In [ ]:
return ProductListResponse(
    data=products,
    page=page
)

- `offset` means how many records you skip
- `limit` means how many records you fetch
<br>
<hr>

In [ ]:
# Create Order API
@app.post("/orders", response_model=OrderResponse, status_code=201)
async def create_order(order: OrderCreate):
    product_ids = [] # store all valid productId 

    for item in order.items:
        try:
            # item have productId and qty
            product_ids.append(ObjectId(item.productId)) 
        except Exception:
            raise HTTPException(status_code=400, detail=f"Invalid productId: {item.productId}")
            # Exception will raised for invalid id (format), not for id that do not exist in DB

    # Fetch product documents using productId
    products = await product_collection.find({"_id": {"$in": product_ids}}).to_list(length=len(product_ids))

    if len(products) != len(order.items):
        # order.items will contain all items i.e, existing or non existing in DB
        # products will conain only items that exist in DB
        found_ids = {str(p["_id"]) for p in products} # set
        missing_ids = [item.productId for item in order.items if item.productId not in found_ids]
        raise HTTPException(status_code=404, detail=f"Products not found: {missing_ids}")

    # Check stock & prepare deduction plan
    stock_updates = []  # collect (product_id, updated_sizes) to update DB later
    # [(ObjectId("64f6c2..."),[{"size": "S", "quantity": 0}, ...]), ...]

    for item in order.items:
        product = next((p for p in products if str(p["_id"]) == item.productId), None)
        
        if not product:
            continue  # Skip if product is None

        sizes = product.get("sizes", []) # get all sizes of that product
        total_stock = sum(size.get("quantity", 0) for size in sizes) # total stock of that product

        if item.qty > total_stock: # if order placed item qty is greater than total stock of that item
            raise HTTPException(
                status_code=400,
                detail=f"Not enough stock for product '{product['name']}' (available: {total_stock}, requested: {item.qty})"
            )

        # Deduct qty from sizes
        # We are not specifying size during order — we’ll deduct from available sizes in any order, starting with the largest available quantity
        
        qty_to_deduct = item.qty # Keeps track of how much quantity is still left to deduct from available sizes
        
        new_sizes = [] # A new list that will replace the current sizes in the product document after stock is updated.
        # it will store [{"size": value, "quatity": updated_quantity}, ...]
        
        # he order doesn’t specify a size, so deduction happens starting with the first size in sizes.
        for size in sizes:
            available = size["quantity"] # get available quatity of a particular size
            
            if qty_to_deduct == 0: # Check if we already deducted enough
                new_sizes.append(size)
                continue

            deduct = min(qty_to_deduct, available)
            new_sizes.append({
                "size": size["size"],
                "quantity": available - deduct
            })
            qty_to_deduct -= deduct

        stock_updates.append((product["_id"], new_sizes))

    # Apply stock updates in MongoDB
    for product_id, updated_sizes in stock_updates:
        await product_collection.update_one(
            {"_id": product_id},
            {"$set": {"sizes": updated_sizes}}
        )

    # Save order in DB
    order_data = {
        "user_id": order.userId,
        "items": [
            {"product_id": item.productId, "quantity": item.qty}
            for item in order.items
        ]
    }

    result = await order_collection.insert_one(order_data)
    
    return {"id": str(result.inserted_id)}

- `order_ids` will store `productId` in `ObjectId` type.
- `product_collection.find({"_id": {"$in": product_ids}})` will return data in `Cursor` type and we converted into a list.
- `(p for p in products if str(p["_id"]) == item.productId)` is generator.
- `next((p for p in products if str(p["_id"]) == item.productId), None)`: Loop through `products`, as soon as it finds a match, it stops and returns that product. If no matching product is found, it returns `None`.
- `next()` is used to retrieve the next item from an iterator. If there are no more items, it raises `StopIteration` — unless you provide a default value (like `None`), in which case it returns that instead.

**Why are we loop thorugh `order.items` to check valid product but already have all valid product in `products`?** Reason is `order.items` contains the original request from the user, which includes the quantity (`qty`) for each `productId`. Whereas products is just a list of existing product documents from MongoDB. So, you need both the product and the quantity requested for that specific product to do proper stock deduction. That's why.

<br>

**Why are we again loop through `order.items` while saving ordered data in `order_collection` as `order.items` may contain products that does not exist in DB?**

Reason is when code start executing and will reach at this portion:

    if len(products) != len(order.items):
        found_ids = {str(p["_id"]) for p in products}
        missing_ids = [item.productId for item in order.items if item.productId not in found_ids]
        raise HTTPException(status_code=404, detail=f"Products not found: {missing_ids}")

If any product in `order.items` was not found in the database, the request is rejected with HTTP 404 Error. So no invalid/nonexistent product will pass this stage.

So by the time we reach this line `result = await order_collection.insert_one(order_data)`, You can be 100% sure that:

    - All productIds in order.items:
        - Are valid ObjectIds
        - Exist in the database
    - Any invalid or missing products have already triggered an HTTPException

In [ ]:
# Get Orders by User ID
@app.get("/orders/{user_id}", response_model=OrderListResponse)
async def get_user_orders(
    user_id: str = Path(...),
    limit: int = Query(10),
    offset: int = Query(0)
):
    pipeline = [
        {"$match": {"user_id": user_id}},
        {"$sort": {"_id": 1}},
        {"$skip": offset},
        {"$limit": limit},
        {"$unwind": "$items"}, # Splits each order document into multiple documents — one per item in the items array
        {
            "$addFields": { # create new field to join with products
                "items.product_id_obj": { # items.product_id_obj means inside items
                    "$toObjectId": "$items.product_id"
                }
            }
        },
        {
            "$lookup": {
                "from": "products",
                "localField": "items.product_id_obj",
                "foreignField": "_id",
                "as": "product_details"
            }
        },
        {"$unwind": "$product_details"},
        {
            "$group": {
                "_id": "$_id",
                "items": {    # not prrvous items, it is newly created by $group
                    "$push": {
                        "productDetails": {
                            "name": "$product_details.name",
                            "id": {"$toString": "$product_details._id"}
                        },
                        "qty": "$items.quantity"
                    }
                },
                "total": {
                    "$sum": {
                        "$multiply": ["$items.quantity", "$product_details.price"]
                    }
                }
            }
        },
        {
            "$project": {
                "id": {"$toString": "$_id"},
                "items": 1,
                "total": 1
            }
        }
    ]


    orders_cursor = order_collection.aggregate(pipeline)
    orders = [order async for order in orders_cursor]

    # Pagination meta
    page_info = {
        "next": offset + limit,
        "limit": len(orders),
        "previous": offset - limit if offset - limit >= 0 else -10
    }

    return {
        "data": orders,
        "page": page_info
    }

1. `$match`: We are fetching user-specific data from the `orders` collection
→ so that we can filter only those orders which belong to the given `user_id`.

2. `$sort`: We are sorting the user’s orders in ascending order based on `_id`
→ so that we can apply consistent pagination and return results in a predictable order.

3. `$skip`: We are skipping the first `offset` number of documents from the sorted results
→ so that we can support pagination by starting from the correct position.

4. `$limit`: We are limiting the number of documents to `limit`
→ so that we only return a specific number of orders as per pagination settings.
     

In [ ]:
# Output:
{
  "_id": ObjectId("64aa1234567890abcdef1234"),
  "user_id": "64bb1234567890abcdef5678",
  "items": [
    {
      "product_id": "64cc1234567890abcdef9999",
      "quantity": 2
    },
    {
      "product_id": "64cc1234567890abcdef8888",
      "quantity": 1
    }
  ]
}


5. `$unwind`: We are deconstructing the `items` array in each order into individual documents
→ so that we can work with each item separately, especially for joining with the `products` collection.   

In [ ]:
# Output:
[
  {
    "_id": ObjectId("64aa1234567890abcdef1234"),
    "user_id": "64bb1234567890abcdef5678",
    "items": {
      "product_id": "64cc1234567890abcdef9999",
      "quantity": 2
    }
  },
  {
    "_id": ObjectId("64aa1234567890abcdef1234"),
    "user_id": "64bb1234567890abcdef5678",
    "items": {
      "product_id": "64cc1234567890abcdef8888",
      "quantity": 1
    }
  }
]


6. `$addFields`: We are converting each `items.product_id` (string) into an ObjectId
→ so that we can match it with `_id` field in the `products` collection during the lookup stage.


In [ ]:
# Output:
[
  {
    "_id": ObjectId("64aa1234567890abcdef1234"),
    "user_id": "64bb1234567890abcdef5678",
    "items": {
      "product_id": "64cc1234567890abcdef9999",
      "quantity": 2,
      "product_id_obj": ObjectId("64cc1234567890abcdef9999")
    }
  },
  {
    "_id": ObjectId("64aa1234567890abcdef1234"),
    "user_id": "64bb1234567890abcdef5678",
    "items": {
      "product_id": "64cc1234567890abcdef8888",
      "quantity": 1,
      "product_id_obj": ObjectId("64cc1234567890abcdef8888")
    }
  }
]


7. `$lookup`: We are joining each item with its full product details from the `products` collection
→ so that we can fetch information like name and price of each product.

In [ ]:
# products collection:
[
  {
    "_id": ObjectId("64cc1234567890abcdef9999"),
    "name": "Keyboard",
    "price": 1000,
    "sizes": [
      {
        "size": "S",
        "quantity": 8
      }
    ]
  },
  {
    "_id": ObjectId("64cc1234567890abcdef8888"),
    "name": "Mouse",
    "price": 500,
    "sizes": [
      {
        "size": "M",
        "quantity": 10
      }
    ]
  }
]


In [ ]:
# Output:
[
  {
    "_id": ObjectId("64aa1234567890abcdef1234"),
    "user_id": "64bb1234567890abcdef5678",
    "items": {
      "product_id": "64cc1234567890abcdef9999",
      "quantity": 2,
      "product_id_obj": ObjectId("64cc1234567890abcdef9999")
    },
    "product_details": [
      {
        "_id": ObjectId("64cc1234567890abcdef9999"),
        "name": "Keyboard",
        "price": 1000,
        "sizes": [
          {
            "size": "S",
            "quantity": 8
          }
        ]
      }
    ]
  },
  {
    "_id": ObjectId("64aa1234567890abcdef1234"),
    "user_id": "64bb1234567890abcdef5678",
    "items": {
      "product_id": "64cc1234567890abcdef8888",
      "quantity": 1,
      "product_id_obj": ObjectId("64cc1234567890abcdef8888")
    },
    "product_details": [
      {
        "_id": ObjectId("64cc1234567890abcdef8888"),
        "name": "Mouse",
        "price": 500,
        "sizes": [
          {
            "size": "M",
            "quantity": 10
          }
        ]
      }
    ]
  }
]


8. `$unwind`: We are flattening the `product_details` array
→ so that we can easily access individual fields like `product_details.name` and `product_details.price`.

In [ ]:
# Output:
[
  {
    "_id": ObjectId("64aa1234567890abcdef1234"),
    "user_id": "64bb1234567890abcdef5678",
    "items": {
      "product_id": "64cc1234567890abcdef9999",
      "quantity": 2,
      "product_id_obj": ObjectId("64cc1234567890abcdef9999")
    },
    "product_details": {
      "_id": ObjectId("64cc1234567890abcdef9999"),
      "name": "Keyboard",
      "price": 1000,
      "sizes": [
        {
          "size": "S",
          "quantity": 8
        }
      ]
    }
  },
  {
    "_id": ObjectId("64aa1234567890abcdef1234"),
    "user_id": "64bb1234567890abcdef5678",
    "items": {
      "product_id": "64cc1234567890abcdef8888",
      "quantity": 1,
      "product_id_obj": ObjectId("64cc1234567890abcdef8888")
    },
    "product_details": {
      "_id": ObjectId("64cc1234567890abcdef8888"),
      "name": "Mouse",
      "price": 500,
      "sizes": [
        {
          "size": "M",
          "quantity": 10
        }
      ]
    }
  }
]


9. `$group`: We are regrouping the split items back into their original order using `_id`
→ so that we can reconstruct the complete order with enriched product info and calculate the total bill.

In [ ]:
# Output:
[
  {
    "_id": ObjectId("64aa1234567890abcdef1234"),
    "items": [
      {
        "productDetails": {
          "name": "Keyboard",
          "id": "64cc1234567890abcdef9999"
        },
        "qty": 2
      },
      {
        "productDetails": {
          "name": "Mouse",
          "id": "64cc1234567890abcdef8888"
        },
        "qty": 1
      }
    ],
    "total": 2500
  }
]


10. `$project`: We are shaping the final output to include the order ID, items list, and total price
→ so that we return a clean and client-friendly response format with only necessary fields.

In [ ]:
# Output:
[
  {
    "id": "64aa1234567890abcdef1234",
    "items": [
      {
        "productDetails": {
          "name": "Keyboard",
          "id": "64cc1234567890abcdef9999"
        },
        "qty": 2
      },
      {
        "productDetails": {
          "name": "Mouse",
          "id": "64cc1234567890abcdef8888"
        },
        "qty": 1
      }
    ],
    "total": 2500
  }
]
